<a href="https://colab.research.google.com/github/yi0747/health-checkup-analysis/blob/version_1/notebook/analysis_ver03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 종합 실습 — 건강검진 데이터 분석 (재현 가능한 노트북)

이 노트북은 **위에서 아래로 한 번에 실행**되도록 구성되어 있습니다.
상단에 환경 설정(설치 · import · 경로 · 난수 고정)을 모아 두고,
그 다음 STEP 0~5 분석이 순서대로 이어집니다.

> 처음 실행 시: `런타임 > 모두 실행` (Ctrl+F9)


---
## 0. 환경 설정 (Setup)

재현성을 위해 **설치 → import → 경로 → 난수 고정**을 노트북 맨 위에 모읍니다.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pwd

%cd /content/drive/MyDrive/강의자료/강의코드/health-checkup-analysis-main

!ls

/content
[Errno 2] No such file or directory: '/content/drive/MyDrive/강의자료/강의코드/health-checkup-analysis-main'
/content
drive  sample_data


In [3]:
# [설치] 필요한 패키지를 첫 코드 셀에서 한 번에 설치
#  - Colab 기본 제공 외 패키지를 여기서 설치해야 새 런타임에서도 재현됩니다.
#  - 분석 함수는 같은 폴더의 utils.py 에서 import 합니다 (아래 import 셀 참고).
!pip install -q scikit-posthocs openpyxl


In [4]:
# [import] 사용하는 라이브러리를 한 셀에 모읍니다
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from sklearn.metrics import roc_auc_score, roc_curve

# 분석·검정·시각화 함수는 utils.py 로 분리해 두었습니다
from utils.utils import (
    plot_box, plot_crosstab,
    run_2group, run_multigroup, posthoc, run_chi2,
)


ModuleNotFoundError: No module named 'utils'

In [ ]:
# [한글 폰트] 그래프에 한글이 깨지지 않도록 설정 (Colab)
!apt-get -qq install fonts-nanum
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)
sns.set_theme(style='whitegrid', palette='colorblind', font='NanumGothic')

In [ ]:
# [경로] 데이터 경로는 코드 곳곳에 직접 쓰지 말고 변수로 관리합니다

BASE_DIR  = '/content/drive/MyDrive/강의자료/강의코드/health-checkup-analysis-main'
DATA_PATH = f'{BASE_DIR}/data/건강검진_실습데이터.xlsx'

In [ ]:
# [난수 고정] 실행할 때마다 결과가 달라지지 않도록 seed 고정
SEED = 42
np.random.seed(SEED)

In [ ]:
# [변수 정의] 분석 전반에서 사용할 변수 목록을 한곳에 정의
num_cols = ['age', 'bmi', 'waist', 'sbp', 'glucose', 'tg', 'hdl', 'crp']
cat_cols = ['sex', 'region', 'smoking', 'exercise_freq', 'metabolic_syndrome']

# 검정에서 설명변수로 쓸 범주형 목록 (결과변수 metabolic_syndrome 제외)
cat_vars = ['sex', 'region', 'smoking', 'exercise_freq']


---
## STEP 0. 데이터 불러오기


In [ ]:
# 데이터 로드 (경로 변수 사용)
df = pd.read_excel(DATA_PATH)
df_raw = df.copy()          # 진단·비교용 원본 보존
df.head()

---
## STEP 1. 데이터 진단 — 구조 · 타입 · 결측


In [ ]:
# 구조 · 타입 · 결측 확인
print(df.shape)
df.info()

In [ ]:
# 연속형 변수 전체 분포 — 히스토그램 + 평균/중앙값 + 왜도
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, c in zip(axes.ravel(), num_cols):
    ax.hist(df[c].dropna(), bins=25, color='#9DB4D4', edgecolor='white')
    ax.axvline(df[c].mean(),   color='#C00000', ls='--', label='평균')
    ax.axvline(df[c].median(), color='#1F3864', ls='-',  label='중앙값')
    ax.set_title(f"{c}  (왜도 {df[c].skew():.2f})")
    ax.legend(fontsize=8)
plt.suptitle('연속형 변수 분포', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# 범주형 변수 전체 분포 — 막대그래프
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, c in zip(axes, cat_cols):
    df[c].value_counts().sort_index().plot(kind='bar', ax=ax, color='#1F3864')
    ax.set_title(c); ax.set_xlabel('')
plt.tight_layout(); plt.show()

In [ ]:
# 결측 비율 확인 (대치 전 원본 기준)
miss = df_raw.isnull().sum()
miss_pct = (miss / len(df_raw) * 100).round(1)
pd.DataFrame({'결측수': miss, '결측%': miss_pct})[miss > 0]

In [ ]:
# 결측 대치 — 중앙값으로 통일
for c in ['hdl', 'crp']:
    df[c] = df[c].fillna(df[c].median())

print('결측 합계:', df.isnull().sum().sum())

---
## STEP 2. 기술통계 — 중심 · 산포 · 분포 요약


In [ ]:
# 연속형 변수 요약 — 중심(평균·중앙값) · 산포(표준편차) · 분포(왜도)
desc = df[num_cols].agg(['mean', 'median', 'std']).T
desc['skew'] = df[num_cols].skew()
desc.round(2)

In [ ]:
# 산포 비교를 위한 변동계수(CV) — 단위가 다른 변수 간 상대적 퍼짐
desc['CV(%)'] = (df[num_cols].std() / df[num_cols].mean() * 100).round(1)
desc.round(2)

In [ ]:
# 범주형 변수 빈도 — 개수와 비율(%)
for c in cat_cols:
    counts = df[c].value_counts().sort_index()
    pct = (counts / len(df) * 100).round(1)
    print(f'── {c} ──')
    print(pd.DataFrame({'개수': counts, '비율(%)': pct}))
    print()

---
## STEP 3. 연속형 검정 — 집단 간 평균 차이


연속형 변수의 집단 간 차이를 봅니다.
`plot_box(df, group_col, num_cols)` 로 박스플롯을,
`run_2group(df, group_col, num_cols)` 로 2집단 검정을 수행합니다.
(함수 정의는 `utils.py` 참고)


In [ ]:
# 그룹: metabolic_syndrome (2집단)
plot_box(df, 'metabolic_syndrome', num_cols)
run_2group(df, 'metabolic_syndrome', num_cols)


3집단 이상은 `run_multigroup(df, group_col, num_cols)` 로 검정하고,
유의한 조합은 `posthoc(df, group_col, y, method)` 로 사후검정합니다.


In [ ]:
# 그룹: exercise_freq (3집단)
plot_box(df, 'exercise_freq', num_cols)
run_multigroup(df, 'exercise_freq', num_cols)


In [ ]:
# 사후검정 예시: exercise_freq × hdl (ANOVA 계열 → Tukey HSD)
posthoc(df, 'exercise_freq', 'hdl', 'ANOVA')


---
## STEP 4. 범주형 검정 — 범주 간 관련성


In [ ]:
# 범주형 일괄 검정 — 여러 변수 × 대사증후군
#   기대빈도 점검 → 카이제곱 / Fisher 자동 선택 + 효과크기(Cramér's V)
run_chi2(df, cat_vars, 'metabolic_syndrome')


In [ ]:
# 교차표 시각화 일괄 — 각 변수별 대사증후군 비율
plot_crosstab(df, cat_vars, 'metabolic_syndrome')


---
## STEP 5. 상관 · 회귀 — 관계와 독립적 효과


In [ ]:
# 상관행렬 — 연속형 변수 간 선형 관계
corr = df[num_cols].corr(method='pearson')

plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True)
plt.title('연속형 변수 상관행렬 (Pearson)')
plt.tight_layout(); plt.show()

In [ ]:
# 범주형 변수를 0/1 더미로 변환
df['male'] = (df['sex'] == '남').astype(int)
df['smk']  = (df['smoking'] == '흡연').astype(int)

# 다중공선성 점검 — VIF
predictors = ['age', 'bmi', 'waist', 'glucose', 'tg', 'hdl', 'crp', 'male', 'smk']
X = add_constant(df[predictors])

vif = pd.DataFrame({
    '변수': predictors,
    'VIF': [variance_inflation_factor(X.values, i + 1) for i in range(len(predictors))]
})
vif.sort_values('VIF', ascending=False).round(2)

In [ ]:
# 로지스틱 회귀 — 모든 변수를 동시에 넣어 서로 보정
model = smf.logit(
    'metabolic_syndrome ~ age + bmi + waist + glucose + tg + hdl + crp + male + smk',
    data=df
).fit()
print(model.summary())

In [ ]:
# 조정 오즈비(adjusted OR)와 95% 신뢰구간 — 해석용 표
res = pd.DataFrame({
    'OR': np.exp(model.params),
    'p': model.pvalues
}).join(np.exp(model.conf_int()).rename(columns={0: 'CI_low', 1: 'CI_high'}))
res['유의'] = np.where(res['p'] < 0.05, '*', '')
res.drop('Intercept').round(3).sort_values('p')

In [ ]:
# 조정 오즈비 시각화 — forest plot
plot_df = res.drop('Intercept').sort_values('OR')
plt.figure(figsize=(7, 5))
plt.errorbar(plot_df['OR'], range(len(plot_df)),
             xerr=[plot_df['OR'] - plot_df['CI_low'],
                   plot_df['CI_high'] - plot_df['OR']],
             fmt='o', color='#1F3864', capsize=3)
plt.axvline(1, color='#C00000', ls='--')
plt.yticks(range(len(plot_df)), plot_df.index)
plt.xscale('log'); plt.xlabel('조정 오즈비 (log scale)')
plt.title('대사증후군 위험요인 — 조정 오즈비'); plt.tight_layout(); plt.show()

In [ ]:
# ROC 곡선 / AUC
used = model.model.data.row_labels
y_true = df.loc[used, 'metabolic_syndrome']
pred = model.predict(df.loc[used])

auc = roc_auc_score(y_true, pred)
fpr, tpr, _ = roc_curve(y_true, pred)

plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('1 - 특이도'); plt.ylabel('민감도')
plt.legend()
plt.show()